In [ ]:
# --- INSTALAÇÕES (execute se for a primeira vez) ---
!pip install pandas torch transformers spacy
!python -m spacy download pt_core_news_lg

import re
import glob
import json
import pandas as pd
import spacy
from transformers import pipeline

# --- CARREGANDO MODELOS ---
print("Carregando modelo SpaCy (pt_core_news_lg)...")
# Usamos um bloco try-except para lidar com o caso do modelo não estar instalado
try:
    nlp = spacy.load("pt_core_news_lg")
except OSError:
    print("Modelo 'pt_core_news_lg' não encontrado. Baixando e instalando...")
    from spacy.cli import download
    download("pt_core_news_lg")
    nlp = spacy.load("pt_core_news_lg")

print("Carregando modelo de Análise de Sentimento...")
sentiment_classifier = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

print("Modelos carregados com sucesso!")

  Using cached spacy-3.8.7-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached murmurhash-1.0.13-cp312-cp312-win_amd64.whl.metadata (2.2 kB)
  Using cached preshed-3.0.10-cp312-cp312-win_amd64.whl.metadata (2.5 kB)
  Using cached thinc-8.3.6-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached srsly-2.5.1-cp312-cp312-win_amd64.whl.metadata (20 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached typer-0.17.4-py3-none-any.whl.metadata (15 kB)
  Using cached langcodes-3.5.0-py3-none-any.whl.metadata (29 kB)
  Using cached language_data-1.3.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached blis-1.3.0-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached confection-0.1.5-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
  Using cached cloudpathlib-0.22.0-


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Dicionário principal com a lógica para cada critério de QA
# Mapeamos o ID do critério para uma lista de palavras-chave ou padrões
QA_CRITERIA = {
    # 1. Abertura / Encerramento
    "1.1_abertura": {
        "keywords": ["comgás", "bom dia", "boa tarde", "boa noite", "meu nome é", "em que posso ajudar"],
        "check_type": "keywords_start", # Verifica no início da transcrição
        "description": "Abertura de contato padrão."
    },
    "1.3_encerramento": {
        "keywords": ["algo mais", "posso ajudar em algo mais", "a comgás agradece", "tenha um bom dia", "tenha uma boa tarde"],
        "check_type": "keywords_end", # Verifica no fim da transcrição
        "description": "Encerramento de contato padrão."
    },
    "1.2_espera": {
        "keywords": ["só um momento", "aguarde um instante", "só um minuto", "vou verificar", "obrigado por aguardar"],
        "check_type": "keywords_any", # Verifica em qualquer lugar
        "description": "Sinalizou procedimento de espera."
    },

    # 2. Condução do Atendimento
    "2.1_empatia": {
        "keywords": ["eu entendo", "compreendo", "sinto muito", "peço desculpas pelo transtorno", "vamos resolver"],
        "check_type": "keywords_any",
        "description": "Demonstrou empatia."
    },
    "2.2_cordialidade": {
        "keywords": ["por favor", "por gentileza", "obrigado", "obrigada", "senhor", "senhora"],
        "check_type": "keywords_any",
        "description": "Tratou o cliente com cordialidade."
    },
    "2.3_dialogo_vicios": {
        "keywords": ["tipo assim", "né", "tá", "mano", "cara", "veio"],
        "check_type": "negative_keywords", # Critério reprova se encontrar essas palavras
        "description": "Diálogo sem vícios de linguagem/gírias."
    },

    # 3. Processos e Procedimentos
    "3.0_identificacao_positiva": {
        "keywords": [r"confirmar seu (cpf|nome|endereço)", r"qual (seu|o) (cpf|nome completo)", r"código de usuário"],
        "check_type": "regex_any",
        "description": "Realizou a identificação positiva."
    },
    "3.4_oportunidades": {
        "keywords": ["comgás virtual", "fatura digital", "conta por e-mail", "débito automático", "aproveitando o contato"],
        "check_type": "keywords_any",
        "description": "Criou oportunidades de negócio / fomentou autosserviço."
    },
    "3.5_pesquisa_satisfacao": {
        "keywords": ["pesquisa de satisfação"],
        "check_type": "keywords_end",
        "description": "Direcionou para a pesquisa de satisfação."
    },

    # 4. NCG / Falhas Graves
    "4.0_etica": {
        "keywords": ["absurdo", "incompetente", "esse sistema", "outro atendente", "fez errado", "droga", "inferno"],
        "check_type": "negative_keywords",
        "description": "Ética (não difamou a empresa ou colegas)."
    },
    "4.4_abertura_os": {
        "keywords": [r"(gerar|abrir|registrar) (a os|uma os|ordem de serviço)", r"número da sua os", r"protocolo da os"],
        "check_type": "regex_any",
        "description": "Mencionou a abertura ou o registro de uma OS."
    },
    "4.5_desconexao_indevida": {
        "keywords": ["alô", "está me ouvindo", "a ligação caiu"],
        "check_type": "keywords_end", # Se o cliente diz isso no final, pode ser um sinal
        "description": "Indício de desconexão indevida (EXPERIMENTAL)."
    }
}

# Adicionamos aqui critérios que são difíceis ou impossíveis de automatizar apenas com a transcrição.
# O script irá marcá-los para revisão manual.
MANUAL_REVIEW_CRITERIA = [
    "2.0", "3.1", "3.2", "3.3", "4.1", "4.2", "4.3", "4.6", "4.7", "4.8", "4.10"
]

In [ ]:
def evaluate_call_quality(transcript: str) -> dict:
    """
    Avalia uma única transcrição com base nos critérios definidos em QA_CRITERIA.

    Retorna:
        Um dicionário com o resultado ('Aprovado' ou 'Reprovado') para cada critério.
    """
    results = {}
    transcript_lower = transcript.lower()
    transcript_len = len(transcript_lower)

    start_slice = transcript_lower[:int(transcript_len * 0.15)]
    end_slice = transcript_lower[-int(transcript_len * 0.15):]

    for key, criteria in QA_CRITERIA.items():
        check_type = criteria["check_type"]
        keywords = criteria["keywords"]
        found = False

        target_text = transcript_lower
        if check_type == "keywords_start":
            target_text = start_slice
        elif check_type == "keywords_end":
            target_text = end_slice

        if "keywords" in check_type:
            if any(keyword in target_text for keyword in keywords):
                found = True
        elif "regex" in check_type:
            if any(re.search(pattern, target_text, re.IGNORECASE) for pattern in keywords):
                found = True

        if check_type == "negative_keywords":
            results[key] = "Reprovado" if found else "Aprovado"
        else:
            results[key] = "Aprovado" if found else "Reprovado"

    return results